# Surface Code

Working on making a nice interface for the d=3 surface codes from Tomita & Svore 2014.

In [ ]:
import sys
sys.path.append('..')

In [ ]:
from loqs.codepacks import d3_surface_code as codepack

## Stabilizer Plaquette Template

One of the nice things about the surface code is that the stabilizers can be described in a tileable way.

<img src="images/TomitaSvoreFig2.png" alt="Tomita & Svore 2014 Fig 2" width="400"/>
<img src="images/TomitaSvoreFig4b.png" alt="Tomita & Svore 2014 Fig 4b" width="400"/>

(Reproduced from Tomita & Svore 2014)

We can take advantage of this by coding up a "template" circuit that we will use to build up the syndrome circuit.
Below we show the `CircuitPlaquetteFactory` object provided in the surface code codepack, which holds these template circuits and is responsible for substituting the correct qubit values into this template.

In [ ]:
# X stabilizer, matching Fig 2a
print(str(codepack.surface_factory.circuit_templates['X']))

In [ ]:
# Z stabilizer, matching Fig 2b
print(str(codepack.surface_factory.circuit_templates['Z']))

In [ ]:
# Alternate Z stabilizer, matching Fig 4b
print(str(codepack.surface_factory.circuit_templates['Zalt']))

The `CircuitPlaquetteFactory` can be used to get the actual circuits by providing the template type and qubit labels. <font color="red"> Note that the qubit labels have to match the template line labels exactly. </font>

In [ ]:
c = codepack.surface_factory.get_circuit("X", ["A0", "D0", "D1", "D2", "D3"])
print(str(c))

There are a few advanced modes that can be used. We can also drop certain CNOTs in the case of lower weight stabilizer checks that maintain the same schedule to avoid CNOT collisions, or the midcircuit measurement can be removed in case one wants to generate the syndrome preparation circuit rather than the syndrome extraction circuit.

In [ ]:
# Pass in Nones to skip some checks
c = codepack.surface_factory.get_circuit("X", ["A0", "D0", None, "D2", None])
print(str(c))

In [ ]:
# Can omit the instrument
c = codepack.surface_factory.get_circuit("X", ["A0", "D0", "D1", "D2", "D3"], omit_gates='Iz')
print(str(c))

## Syndrome Circuit Generation

A syndrome circuit can now be quickly specified as a `PlaquetteCircuit` by giving the factory (from above) and the stabilizer types and qubit lists, which is essentially the information provided in the figure below.

<img src="images/TomitaSvoreFig1.png" alt="Tomita & Svore 2014 Fig 1" width="400"/>

(Reproduced from Tomita & Svore 2014)

Below we show examples of the stabilizer specifications given in Surface-17 and Surface-13.
In Surface-17, all stabilizers can be done at once because each stabilizer has a dedicated auxiliary qubit. In Surface-13, there are two phases: the weight-4 checks can be done in parallel, as can the weight-2, but they reuse the same auxiliary qubits. That is specified as well.

In [ ]:
# All checks in one stage for Surface-17
syndrome_circ_surface17 = codepack.surface17_syndrome
syndrome_circ_surface17.spec.stage_specs

In [ ]:
# Two stages for Surface-13
syndrome_circ_surface13 = codepack.surface13_syndrome
syndrome_circ_surface13.spec.stage_specs

Finally, we can generate the full syndrome extraction circuits.

In [ ]:
# Single stage for Surface-17
c17 = syndrome_circ_surface17.get_circuit()
print(str(c17))

In [ ]:
c13 = syndrome_circ_surface13.get_circuit()

# We can see the two phases of Surface-13
print(str(c13))

We can also do some advanced things. We can ask pyGSTi to parallelize/compress the circuit if we wanted (we likely want to do this manually, but hey the capability is there), or we can remove the midcircuit measurements to get the syndrome preparation circuit (could be useful for Harper&Flammia 2023-style benchmarking where 2 syndrome preps = synthetic idle).

In [ ]:
# Can compress the entire circuit
c = syndrome_circ_surface13.get_circuit(compress=True)
print(str(c))

In [ ]:
# Or only compress within stages
c = syndrome_circ_surface13.get_circuit(compress=True,
                                        compress_stages_only=True)
print(str(c))

In [ ]:
c = syndrome_circ_surface17.get_circuit(omit_gates='Iz')
print(str(c))